In [ ]:
# libraries
import os
import sys
import json
import hashlib
import subprocess
import shlex
import argparse
import fcntl
from pathlib import Path
import nbformat
import pandas as pd
from IPython.display import display


In [ ]:
# controller imports; scientific source and its hash stay unchanged
REPO = Path.cwd()
PIPE = {}
for cell in nbformat.read(REPO / 'pipeline_tools.ipynb', 4).cells:
    exec(compile(cell.source, 'pipeline_tools.ipynb', 'exec'), PIPE)


In [ ]:
# original completion contracts; only incomplete compatible work is scheduled
def train_result(root, job, cfg, runtime, data_hash, code_hash):
    folder = root / 'd_phase1_train/output/phase1_runs' / job['run']
    path = folder / 'result.json'
    if not path.is_file():
        return None
    result = json.loads(path.read_text())
    spec = next(s for s in cfg['models'] if s['name'] == job['model'])
    signature = runtime['run_signature'](cfg, spec, 1, job['regime'], job['replicate'], data_hash, code_hash)
    if (result['signature'], result['optimizer_steps'], result['model'], result['regime'], result['replicate'], result['phase']) != (signature, cfg['steps1'], job['model'], job['regime'], job['replicate'], 1):
        raise ValueError('incompatible training result: ' + job['run'])
    if not (folder / 'best.pt').is_file():
        raise ValueError('missing best checkpoint: ' + job['run'])
    return result

def evaluated(root, job, trained):
    folder = root / 'e_phase1_test/output/eval_runs' / job['run']
    path = folder / 'completion.json'
    if not path.is_file():
        return False
    result = json.loads(path.read_text())
    checkpoint = root / 'd_phase1_train/output/phase1_runs' / trained['checkpoint']
    if (result['run'], result['index'], result['training_signature']) != (job['run'], job['index'], trained['signature']):
        raise ValueError('incompatible evaluation: ' + job['run'])
    if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != result['checkpoint_sha256']:
        raise ValueError('changed evaluation checkpoint: ' + job['run'])
    for name, digest in result['files'].items():
        if hashlib.sha256((folder / name).read_bytes()).hexdigest() != digest:
            raise ValueError('changed evaluation artifact: ' + str(folder / name))
    return True

def runtime_inputs(root):
    source = root / 'd_phase1_train/input'
    cfg = json.loads((source / 'config.json').read_text())
    runtime = {}
    for cell in nbformat.read(source / 'runtime.ipynb', 4).cells:
        exec(compile(cell.source, 'runtime.ipynb', 'exec'), runtime)
    return cfg, runtime, hashlib.sha256((source / 'dataset.pt').read_bytes()).hexdigest(), (source / 'runtime.sha256').read_text()

def active_submissions(records, own_job):
    # fail closed if Slurm cannot be queried; ignore this dispatch controller only
    output = subprocess.check_output(['squeue', '--noheader', '--user', os.environ['USER'], '--format=%F'], text=True)
    active = set(output.split())
    known = {str(j['job_id']) for record in records for j in record.get('submitted', [])}
    return sorted((known & active) - {own_job})


In [ ]:
# bounded rolling submissions; one array plus one dependent continuation at a time
def dispatch(args, pipe=PIPE):
    root, control = pipe['load_control'](REPO, args.experiment)
    if not (root / 'ready.json').is_file():
        raise ValueError('preparation incomplete; do not recover this experiment automatically')
    logs = REPO / '.local/slurm_logs' / args.experiment
    logs.mkdir(parents=True, exist_ok=True)
    journal = logs / 'rolling.json'
    with (logs / '.rolling.lock').open('a') as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        record = json.loads(journal.read_text()) if journal.is_file() else {'submitted': [], 'commands': []}
        original = logs / 'submission.json'
        records = [record] + ([json.loads(original.read_text())] if original.is_file() else [])
        active = active_submissions(records, os.environ.get('SLURM_JOB_ID', ''))
        if active:
            raise RuntimeError('experiment jobs already active: ' + ', '.join(active))
        cfg, runtime, data_hash, code_hash = runtime_inputs(root)
        if cfg.get('task') != 'communication_study' or cfg['steps2']:
            raise ValueError('rolling recovery is Phase 1 communication-study only')
        jobs = pd.read_csv(root / 'job_matrix.csv').to_dict('records')
        results = {j['index']: train_result(root, j, cfg, runtime, data_hash, code_hash) for j in jobs}
        missing = [j['index'] for j in jobs if results[j['index']] is None]
        stage = 'train1' if missing else 'evaluate1'
        if not missing:
            missing = [j['index'] for j in jobs if not evaluated(root, j, results[j['index']])]
        # afterany releases the controller even on failure; stop rather than silently looping retries
        if args.after_batch:
            batch = json.loads(Path(args.after_batch).read_text())
            incomplete = [i for i in batch['indices'] if (results[i] is None if batch['stage'] == 'train1' else not evaluated(root, jobs[i], results[i]))]
            if incomplete:
                raise RuntimeError('previous batch incomplete; inspect its Slurm logs, then rerun this launcher: ' + str(incomplete))
        display(pd.DataFrame([{'expected_runs': len(jobs), 'training_complete': sum(v is not None for v in results.values()),
                              'next_stage': stage, 'remaining_in_stage': len(missing), 'next_batch': min(len(missing), args.chunk_size)}]))
        if not missing:
            if not args.dry_run:
                pipe['finish_study'](root, control)
                record['complete'] = True
                pipe['atomic_json'](record, journal)
            return
        batch = {'stage': stage, 'indices': missing[:args.chunk_size]}
        sequence = len(record['submitted'])
        batch_path = logs / f'rolling-batch-{sequence:05d}.json'
        common = ['sbatch', '--parsable', '--partition=nvgpu', '--gpus=1', '--constraint=GPU_ANY', '--cpus-per-task=2',
                  '--mem=32G', '--time=' + args.time, '--export=ALL', '--chdir=' + str(REPO), '--output=' + str(logs / '%x-%A_%a.out')]
        if args.account:
            common += ['--account=' + args.account]
        environment = {**os.environ, 'NCA_PYTHON': sys.executable}
        launcher = [str(REPO / 'slurm/rolling.sh'), '--experiment', args.experiment, '--max-parallel', str(args.max_parallel),
                    '--chunk-size', str(args.chunk_size), '--time', args.time]
        if args.account:
            launcher += ['--account', args.account]
        if not args.dry_run:
            if stage == 'evaluate1':
                pipe['prepare_study_evaluation'](root)
            pipe['atomic_json'](batch, batch_path)
        def submit(command, label):
            record['commands'].append(command)
            if args.dry_run:
                print(shlex.join(command))
                return 'DRY'
            try:
                output = subprocess.check_output(command, cwd=REPO, env=environment, text=True, stderr=subprocess.STDOUT).strip()
                job_id = output.split(';')[0]
                if not job_id.isdigit():
                    raise RuntimeError('unexpected sbatch output: ' + output)
                record['submitted'].append({'stage': label, 'job_id': job_id})
                return job_id
            except subprocess.CalledProcessError as error:
                record['submission_error'] = error.output
                raise RuntimeError(error.output) from error
            finally:
                pipe['atomic_json'](record, journal)
        array = common + ['--job-name=nca-roll-' + stage, f'--array=0-{len(batch["indices"]) - 1}%{args.max_parallel}'] + launcher + ['--batch', str(batch_path)]
        parent = submit(array, stage)
        submit(common + ['--job-name=nca-roll-dispatch', '--dependency=afterany:' + parent] + launcher + ['--after-batch', str(batch_path)], 'dispatch')
        display(pd.DataFrame(record['submitted']).tail(2))

def main(arguments=None):
    parser = argparse.ArgumentParser(description='Resume a prepared study with bounded Slurm submissions; scientific source is unchanged')
    parser.add_argument('--experiment', required=True)
    parser.add_argument('--max-parallel', type=int, default=8)
    parser.add_argument('--chunk-size', type=int, default=200)
    parser.add_argument('--time', default='12:00:00')
    parser.add_argument('--account', default='')
    parser.add_argument('--batch')
    parser.add_argument('--after-batch')
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(arguments)
    if not 1 <= args.chunk_size <= 500 or args.max_parallel < 1:
        parser.error('chunk-size must be 1..500 and max-parallel positive')
    if args.batch:
        batch = json.loads(Path(args.batch).read_text())
        index = batch['indices'][int(os.environ['SLURM_ARRAY_TASK_ID'])]
        root, control = PIPE['load_control'](REPO, args.experiment)
        command = ['bash', str(REPO / 'slurm/run.sh'), 'worker', '--experiment', args.experiment, '--profile', control['profile'],
                   '--action', batch['stage'], '--index', str(index), '--resume']
        if control.get('sprite'):
            command += ['--sprite', control['sprite']]
        subprocess.check_call(command, cwd=REPO)
        return
    dispatch(args)
